**MGMT298D: Science and Strategy of AI**
# Week 5: Transfer Learning

#### This notebook builds a handbags-vs-shoes image classifier using the HODL (Head-Only Deep Learning) approach to transfer learning. We compare a CNN trained from scratch against one that freezes a pretrained ImageNet backbone and only trains a small classification head on top.

---
# 1 Setup & Data

#### We download a small handbags-vs-shoes dataset and split each class into 50 training, 25 validation, and 25 test images. Images are resized to 224×224 so they match what the pretrained ImageNet models expect.

In [ ]:
import os, shutil, pathlib
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input

keras.utils.set_random_seed(42)

In [ ]:
# Download the handbags-vs-shoes dataset and build train/val/test splits
!wget -q -nc https://www.dropbox.com/s/w07liww46kgxo1m/handbags-shoes.zip
!unzip -qq -n handbags-shoes.zip

base_dir = pathlib.Path('handbags-shoes')
for category in ('handbags', 'shoes'):
    fnames = sorted(os.listdir(base_dir / category))
    for split, sl in [('train', slice(0, 50)), ('validation', slice(50, 75)), ('test', slice(75, 100))]:
        dst = base_dir / split / category
        os.makedirs(dst, exist_ok=True)
        for fn in fnames[sl]:
            if not (dst / fn).exists():
                shutil.copyfile(base_dir / category / fn, dst / fn)

train_ds = keras.utils.image_dataset_from_directory(base_dir/'train',      image_size=(224, 224), batch_size=32, label_mode='binary')
val_ds   = keras.utils.image_dataset_from_directory(base_dir/'validation', image_size=(224, 224), batch_size=32, label_mode='binary')
test_ds  = keras.utils.image_dataset_from_directory(base_dir/'test',       image_size=(224, 224), batch_size=32, label_mode='binary')

# Quick look at a few training images
plt.figure(figsize=(8, 3))
for imgs, labels in train_ds.take(1):
    for i in range(6):
        plt.subplot(1, 6, i+1)
        plt.imshow(imgs[i].numpy().astype('uint8'))
        plt.title('shoe' if labels[i]==1 else 'handbag', fontsize=9)
        plt.axis('off')
plt.tight_layout(); plt.show()

---
# 2 CNN From Scratch (Baseline)

#### A small CNN trained end-to-end on only 100 images. With so few examples and millions of parameters learned from zero, we expect it to struggle. This sets the bar we want transfer learning to beat.

In [ ]:
scratch_cnn = models.Sequential([
    layers.Rescaling(1./255, input_shape=(224, 224, 3)),
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

scratch_cnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
scratch_cnn.fit(train_ds, epochs=10, validation_data=val_ds, verbose=0)

test_acc_scratch = scratch_cnn.evaluate(test_ds, verbose=0)[1]
print(f'From-scratch CNN test accuracy: {test_acc_scratch:.4f}')

---
# 3 HODL — Head-Only Deep Learning

#### Rather than learning filters from scratch, we reuse VGG16's ImageNet-trained convolutional base as a fixed feature extractor and train only a small dense *head* on top. Every image is pushed once through the frozen base, and we save the resulting 7×7×512 feature tensors — so training the head is fast.

In [ ]:
# Frozen pretrained backbone: VGG16 without its classifier head
vgg_base = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
vgg_base.trainable = False

# Push each image through the frozen base once and cache the features
def extract_features(ds):
    feats, labs = [], []
    for imgs, y in ds:
        feats.append(vgg_base.predict(preprocess_input(imgs), verbose=0))
        labs.append(y.numpy())
    return np.concatenate(feats), np.concatenate(labs)

train_feats, train_labels = extract_features(train_ds)
val_feats,   val_labels   = extract_features(val_ds)
test_feats,  test_labels  = extract_features(test_ds)

print(f'Extracted feature shape: {train_feats.shape}  (samples, 7, 7, 512)')

In [ ]:
hodl_head = models.Sequential([
    layers.Flatten(input_shape=(7, 7, 512)),
    layers.Dense(128, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

hodl_head.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
hodl_head.fit(train_feats, train_labels, epochs=10, batch_size=32,
              validation_data=(val_feats, val_labels), verbose=0)

test_acc_hodl = hodl_head.evaluate(test_feats, test_labels, verbose=0)[1]
print(f'HODL test accuracy: {test_acc_hodl:.4f}')

---
# 4 Model Comparison

#### Side-by-side comparison of the from-scratch CNN against HODL on test accuracy.

In [ ]:
print('Model Comparison:')
print('-' * 40)
print(f'From-scratch CNN:   {test_acc_scratch:.4f}')
print(f'HODL:               {test_acc_hodl:.4f}')
print('-' * 40)

plt.bar(['From-scratch CNN', 'HODL'], [test_acc_scratch, test_acc_hodl])
plt.ylabel('Test Accuracy')
plt.title('Transfer Learning vs From Scratch')
plt.ylim(0.4, 1.02)
plt.show()